In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_sales = spark.table(f"{catalogo}.{esquema_source}.everyday_Sales")
df_items = spark.table(f"{catalogo}.{esquema_source}.items")

In [0]:
df_sales = (
    df_sales
        .dropna(how="all")
        .filter(
            col("ItemCode").isNotNull() &
            (trim(col("ItemCode")) != "")
        )
)

In [0]:
df_items = (
    df_items
        .dropna(how="all")
        .filter(
            col("ItemCode").isNotNull() &
            (trim(col("ItemCode")) != "")
        )
)

In [0]:
df_joined = (
    df_sales.alias("s")
    .join(
        df_items.alias("i"),
        col("s.ItemCode") == col("i.ItemCode"),
        "inner"
    )
    .select(
        col("s.Date"),
        col("s.Time"),
        col("s.ItemCode"),
        col("i.ItemName"),
        col("i.CategoryCode"),
        col("i.CategoryName"),
        col("s.QuantitySold_kilo"),
        col("s.UnitSellingPrice"),
        col("s.SaleOrReturn"),
        col("s.Discount")
    )
)

In [0]:
df_updated = df_joined  

In [0]:
df_updated.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.sales_cleaned_grouped")

In [0]:
df_sales.printSchema()
df_items.printSchema()

In [0]:
display(df_updated)